# NaN 값 제거 및 'df_clean' 생성

In [ ]:
import pandas as pd
import warnings
import os

warnings.filterwarnings('ignore') # 경고 메시지 숨김

# =========================================================
# 1. Google Drive 마운트
# Colab에서 Google Drive에 접근하기 위해 필수적인 단계입니다.
# 실행 후 나오는 지시에 따라 인증을 완료해 주세요.
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# 파일 경로 설정 (사용자님의 파일이 Drive 루트 폴더에 있다고 가정합니다.)
# 실제 파일 경로에 따라 'MyDrive' 뒤의 경로를 수정해야 할 수 있습니다.
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"
# 만약 파일이 루트가 아닌 다른 폴더에 있다면:
# 예: FILE_PATH = "/content/drive/MyDrive/연구데이터/processed_analysis_data.csv"


# 2. 파일 로드 (요청하신 정확한 파일명 사용)
try:
    # 경로를 확인하고 파일 로드
    if not os.path.exists(FILE_PATH):
        # Drive 루트에서 파일을 찾지 못했을 경우, Colab 기본 경로에서 시도
        # (이전에 업로드된 파일의 경우)
        FILE_PATH_LOCAL = "processed_analysis_data.csv"
        df = pd.read_csv(FILE_PATH_LOCAL)
        print(f"파일을 Drive가 아닌 로컬 경로 ({FILE_PATH_LOCAL})에서 로드했습니다.")
    else:
        df = pd.read_csv(FILE_PATH)
        print(f"파일을 Drive 경로 ({FILE_PATH})에서 로드했습니다.")


except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 파일 경로 또는 이름을 확인해 주세요: {e}")
    # 파일 로드 실패 시 코드 실행을 중단합니다.
    exit()

# 3. 분석에 사용할 모든 핵심 변수 리스트 정의
REQUIRED_COLUMNS = [
    'sentiment_index_bert', # SI_t
    'KOSPI_등락률',        # R_t
    'R_t+1',              # R_t+1 (다음 날 수익률)
    'log_Volume_t',       # log_Volume_t (거래량)
    'Volatility_t',       # Volatility_t (당일 변동성)
    'Volatility_t+1'      # Volatility_t+1 (다음 날 변동성)
]

# 4. 필요한 컬럼만 추출하여 NaN이 있는 행 전체를 제거 (Final Cleaning)
# 이 과정이 모든 통계 모델링 전에 데이터 일관성을 확보하는 핵심입니다.
df_clean = df[REQUIRED_COLUMNS].dropna()

# 5. 분석의 편의를 위해 컬럼 이름 정리
df_clean.columns = [
    'SI_t',
    'R_t',
    'R_t+1',
    'log_Volume_t',
    'Volatility_t',
    'Volatility_t+1'
]

# 6. 결과 확인
original_rows = len(df)
final_rows = len(df_clean)
removed_rows = original_rows - final_rows

print("\n=======================================================")
print("✅ NaN 값 제거 및 'df_clean' 생성 결과")
print("=======================================================")
print(f"원본 데이터 행 수: {original_rows} 행")
print(f"NaN 값 제거 후 'df_clean' 행 수: {final_rows} 행")
print(f"제거된 행 수: {removed_rows} 행")
print("\n[결론] 통계 분석을 위한 최종 클린 데이터셋 'df_clean'이 준비되었습니다.")
print("=======================================================")

# df_clean의 시작 부분 출력 (확인용)
print("\n[df_clean 상위 5개 행]")
print(df_clean.head())

In [ ]:
import pandas as pd

# 1. 수치 확인을 위한 계산
total_articles = 68243  # 중복 제거 후 기사 총수
initial_days = 1456     # 원본 데이터 행 수
final_days = len(df_clean) # df_clean 행 수 (1451)
removed_days = initial_days - final_days

# 하방 위험일(Downside days) 계산 (R_t <= 0 기준)
downside_days = len(df_clean[df_clean['R_t'] <= 0])
downside_ratio = (downside_days / final_days) * 100

# 일평균 기사 수
avg_articles = total_articles / final_days

# 2. 결과 출력
print("=======================================================")
print("📊 논문 3장 수치 최종 검증 리포트 (df_clean 기준)")
print("=======================================================")
print(f"1. 최종 분석 관측치 (N): {final_days} observations")
print(f"2. 제거된 행 수 및 이유: {removed_days} days (Due to time-lag construction)")
print(f"3. 일평균 뉴스 기사 수: {avg_articles:.22f} articles")
print(f"4. 하방 위험일 수 (R_t <= 0): {downside_days} days")
print(f"5. 하방 위험일 비율: {downside_ratio:.2f}%")
print("=======================================================")

# 3. 영문 표 출력을 위한 데이터프레임
table_data = {
    "Item": ["Initial Trading Days", "Excluded (Time-lag)", "Final Observations (N)",
             "Avg. Articles per Day", "Downside Ratio (R<=0)"],
    "Value": [initial_days, removed_days, final_days,
              round(avg_articles, 2), f"{downside_ratio:.2f}%"]
}
df_table = pd.DataFrame(table_data)
print("\n[Table 2. Quantitative Summary Data]")
print(df_table.to_string(index=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 상관관계 행렬 계산
corr = df_clean.corr()

# 2. 히트맵 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt=".3f")
plt.title("Correlation Matrix: Sentiment & Market Variables")
plt.show()

print("\n[해석 가이드]")
print("1. SI_t와 R_t+1의 상관계수: 뉴스 심리가 내일 주가 방향과 일치하는지 확인")
print("2. SI_t와 log_Volume_t: 뉴스 심리가 거래 활성도에 영향을 주는지 확인")

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# 최대 시차(Max Lag)를 5일(1주일 거래일)로 설정하여 검정
print("📊 [Granger Causality Test: SI_t -> R_t+1]")
results = grangercausalitytests(df_clean[['R_t+1', 'SI_t']], maxlag=5, verbose=True)

print("\n[해석 방법]")
print("- p-value가 0.05보다 작으면: '뉴스 심리가 주가 수익률을 Granger-Cause 한다' (예측력이 있다)고 판단합니다.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import grangercausalitytests

# 1. 상관계수 히트맵 (변수 간 관계 파악)
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean[['SI_t', 'R_t', 'log_Volume_t', 'Volatility_t']].corr(),
            annot=True, cmap='RdBu_r', center=0)
plt.title("Variable Correlation Matrix")
plt.show()

# 2. [수정된] Granger 인과성 검정
# R_t+1 대신 R_t를 사용하여 '1일 전 뉴스 -> 오늘 주가'의 매커니즘을 검정합니다.
print("\n📊 [Granger Causality Test: SI_t -> R_t (Standard)]")
# 결과에서 Lag 1이 유의미하다면 "오늘의 뉴스가 내일의 주가에 영향을 준다"는 뜻입니다.
granger_results = grangercausalitytests(df_clean[['R_t', 'SI_t']], maxlag=5, verbose=True)

# 3. 추가 검정: 뉴스 심리가 '거래량'에는 영향을 주는가?
print("\n📊 [Granger Causality Test: SI_t -> log_Volume_t]")
granger_vol = grangercausalitytests(df_clean[['log_Volume_t', 'SI_t']], maxlag=5, verbose=True)

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# df_clean이 이전 단계에서 성공적으로 생성되었음을 가정합니다.
# 컬럼: 'SI_t', 'R_t', 'R_t+1', 'log_Volume_t', 'Volatility_t', 'Volatility_t+1'

# 1. SI 3분위수 (Tertile) 생성
# cut 함수를 사용하여 SI_t를 3개의 동일 크기 그룹으로 나눕니다.
# labels=['SI_Low', 'SI_Mid', 'SI_High']로 범주 이름을 부여합니다.
# include_lowest=True: 가장 낮은 값을 포함합니다.
df_clean['SI_Tertile'] = pd.cut(
    df_clean['SI_t'],
    bins=df_clean['SI_t'].quantile([0, 1/3, 2/3, 1.0]),
    labels=['SI_Low', 'SI_Mid', 'SI_High'],
    include_lowest=True,
    duplicates='drop' # 경계값이 중복될 경우, 중복 레이블을 제거합니다.
)

# 2. R_t 방향 (0: 하락/보합, 1: 상승)
df_clean['R_Direction'] = (df_clean['R_t'] > 0).astype(int)

# 3. 교차표 (Contingency Table) 생성
crosstab_table_tertile = pd.crosstab(df_clean['SI_Tertile'], df_clean['R_Direction'],
                                    rownames=['SI 강도 (3분위수)'],
                                    colnames=['KOSPI 등락 (0:하락/보합, 1:상승)'])

# 4. 카이제곱 (Chi-square) 검정 실행
# H0: 두 변수(SI 강도와 R 방향)는 독립이다 (관련이 없다).
chi2, p_value, dof, expected = chi2_contingency(crosstab_table_tertile)

# 5. 결과 출력
print("=======================================================")
print("✅ SI 3분위수 기반 카이제곱 (Chi-square) 검정 결과")
print("=======================================================")
print("\n--- 1. 교차표 (Contingency Table) ---")
print(crosstab_table_tertile)

print("\n--- 2. 카이제곱 (Chi-square) 검정 결과 ---")
print(f"Chi-square 통계량: {chi2:.4f}")
print(f"P-value: {p_value:.5f}")
print(f"자유도 (DOF): {dof}")

# 6. 최종 결론
if p_value < 0.05:
    print("\n[결론] P-value가 0.05보다 작으므로, 귀무가설(독립)을 기각합니다.")
    print("      **SI 강도와 KOSPI 등락 방향은 통계적으로 유의미한 관련성이 있습니다.**")
else:
    print("\n[결론] P-value가 0.05보다 크므로, 귀무가설(독립)을 기각할 수 없습니다.")
    print("      SI 강도와 KOSPI 등락 방향은 통계적으로 독립입니다 (관련성이 낮습니다).")
print("=======================================================")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os
from google.colab import drive

warnings.filterwarnings('ignore')

# =========================================================
# 1. Google Drive 마운트 및 파일 로드 (이전 경로 사용)
# =========================================================
# 마운트 및 파일 경로는 이전 단계에서 확인되었습니다.

FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    # 파일을 로드합니다.
    df = pd.read_csv(FILE_PATH)
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# 2. 분석에 사용할 모든 핵심 변수 리스트 정의 (감성 구성 요소 포함)
REQUIRED_COLUMNS_X = [
    'sentiment_index_bert',
    'KOSPI_등락률',
    'R_t+1',
    'log_Volume_t',
    'Volatility_t',
    'Volatility_t+1',
    'avg_pos_score',
    'avg_neg_score',
    'avg_neu_score'
]

# 3. 필요한 컬럼만 추출하여 NaN이 있는 행 전체를 제거
df_clean = df[REQUIRED_COLUMNS_X].dropna()

# 4. 분석의 편의를 위해 컬럼 이름 정리
df_clean.columns = [
    'SI_t', 'R_t', 'R_t+1', 'log_Volume_t', 'Volatility_t', 'Volatility_t+1',
    'Pos_Score_t', 'Neg_Score_t', 'Neu_Score_t'
]

print(f"교차 검증용 'df_clean' 데이터셋 준비 완료. 행 수: {len(df_clean)}")
# =========================================================


# 5. 극성 감성 지수 (Polar SI) 생성: Pos_Score - Neg_Score
df_clean['Polar_SI'] = df_clean['Pos_Score_t'] - df_clean['Neg_Score_t']

# 6. Polar SI 3분위수 (Tertile) 생성 (강도 분석을 위해 3분위수로 나눔)
df_clean['Polar_SI_Tertile'] = pd.cut(
    df_clean['Polar_SI'],
    bins=df_clean['Polar_SI'].quantile([0, 1/3, 2/3, 1.0]),
    labels=['P_SI_Low (부정 강함)', 'P_SI_Mid (중립)', 'P_SI_High (긍정 강함)'],
    include_lowest=True,
    duplicates='drop'
)

# 7. R_t 방향 (0: 하락/보합, 1: 상승)
df_clean['R_Direction'] = (df_clean['R_t'] > 0).astype(int)

# 8. 교차표 (Contingency Table) 생성
crosstab_table_polar = pd.crosstab(df_clean['Polar_SI_Tertile'], df_clean['R_Direction'],
                                    rownames=['Polar SI 강도 (3분위수)'],
                                    colnames=['KOSPI 등락 (0:하락/보합, 1:상승)'])

# 9. 카이제곱 (Chi-square) 검정 실행
# H0: 두 변수(Polar SI 강도와 R 방향)는 독립이다 (관련이 없다).
chi2, p_value, dof, expected = chi2_contingency(crosstab_table_polar)

# 10. 결과 출력
print("\n=======================================================")
print("✅ Polar SI (Pos-Neg) 3분위수 기반 카이제곱 (Chi-square) 검정 결과")
print("=======================================================")

print("\n--- 1. 교차표 (Contingency Table) ---")
print(crosstab_table_polar)

print("\n--- 2. 카이제곱 (Chi-square) 검정 결과 ---")
print(f"Chi-square 통계량: {chi2:.4f}")
print(f"P-value: {p_value:.5f}")
print(f"자유도 (DOF): {dof}")

# 11. 최종 결론
if p_value < 0.05:
    print("\n[결론] P-value가 0.05보다 작으므로, 귀무가설(독립)을 기각합니다.")
    print("      **Polar SI 강도와 KOSPI 등락 방향은 통계적으로 유의미한 관련성이 있습니다.**")
else:
    print("\n[결론] P-value가 0.05보다 크므로, 귀무가설(독립)을 기각할 수 없습니다.")
    print("      Polar SI 강도와 KOSPI 등락 방향은 통계적으로 독립입니다 (관련성이 낮습니다).")
print("=======================================================")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# df_clean이 성공적으로 생성되었음을 가정합니다.
# 컬럼: 'SI_t', 'R_t', 'R_t+1', ...

# 1. SI 3분위수 (Tertile) 생성 (이전 단계와 동일)
df_clean['SI_Tertile'] = pd.cut(
    df_clean['SI_t'],
    bins=df_clean['SI_t'].quantile([0, 1/3, 2/3, 1.0]),
    labels=['SI_Low', 'SI_Mid', 'SI_High'],
    include_lowest=True,
    duplicates='drop'
)

# 2. R_t+1의 극단적인 움직임 (Outlier) 정의 및 범주 생성
r_t_plus_1_std = df_clean['R_t+1'].std()
OUTLIER_THRESHOLD = 1.5 * r_t_plus_1_std # 1.5 * Standard Deviation 기준

def categorize_outlier(r_value):
    if abs(r_value) > OUTLIER_THRESHOLD:
        return 1 # 극단 (Outlier)
    else:
        return 0 # 정상 (Normal)

df_clean['R_t+1_Outlier'] = df_clean['R_t+1'].apply(categorize_outlier)
num_outliers = df_clean['R_t+1_Outlier'].sum()
print(f"R_t+1 극단값 Threshold: ±{OUTLIER_THRESHOLD:.5f} (전체 {len(df_clean)}개 중 {num_outliers}개 극단값)")


# 3. 교차표 (Contingency Table) 생성
crosstab_table_outlier = pd.crosstab(df_clean['SI_Tertile'], df_clean['R_t+1_Outlier'],
                                    rownames=['SI 강도 (3분위수)'],
                                    colnames=['R_t+1 (0:정상, 1:극단)'])

# 4. 카이제곱 (Chi-square) 검정 실행
# H0: 두 변수(SI 강도와 R_t+1 극단값 발생)는 독립이다 (관련이 없다).
chi2, p_value, dof, expected = chi2_contingency(crosstab_table_outlier)

# 5. 결과 출력
print("\n=======================================================")
print("✅ SI 3분위수 vs. R_t+1 극단값 카이제곱 (Chi-square) 검정 결과")
print("=======================================================")
print("\n--- 1. 교차표 (Contingency Table) ---")
print(crosstab_table_outlier)

print("\n--- 2. 카이제곱 (Chi-square) 검정 결과 ---")
print(f"Chi-square 통계량: {chi2:.4f}")
print(f"P-value: {p_value:.5f}")
print(f"자유도 (DOF): {dof}")

# 6. 최종 결론
if p_value < 0.05:
    print("\n[결론] P-value가 0.05보다 작으므로, 귀무가설(독립)을 기각합니다.")
    print("      **SI 강도는 다음 날 KOSPI의 극단적인 움직임 발생 확률과 유의미한 관련성이 있습니다.**")
else:
    print("\n[결론] P-value가 0.05보다 크므로, 귀무가설(독립)을 기각할 수 없습니다.")
    print("      SI 강도는 다음 날 KOSPI의 극단적인 움직임 발생 확률과 관련성이 낮습니다.")
print("=======================================================")

# $\text{NaN}$ 처리를 보완하여 실행한 $\text{SI}$ (중앙값 기준)와 다음 날 수익률 방향 ($\text{R}_{t+1}$) 간의 카이제곱 검정 결과

In [ ]:
import pandas as pd
import os
from scipy.stats import chi2_contingency
from google.colab import drive
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Google Drive 재마운트 (마운트 성공 가정)
print("--- 1. Google Drive 재마운트를 시도합니다. ---")
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive 마운트 성공.")
except Exception as e:
    print(f"❌ Drive 마운트 실패: {e}")
    exit()

# 파일 설정 및 로드 (경로 재확인)
INPUT_DIR = "/content/drive/MyDrive/data"
ANALYSIS_FILE = "processed_analysis_data.csv"
ANALYSIS_PATH = os.path.join(INPUT_DIR, ANALYSIS_FILE)

try:
    df_analysis = pd.read_csv(ANALYSIS_PATH)
    print(f"✅ 파일 로드 성공: {ANALYSIS_PATH}")

    # ========================================================
    # 2. 필수 변수만 추출하고 NaN 값 제거 (가장 중요한 수정 부분)
    # ========================================================
    REQUIRED_COLS_CHI2 = ['sentiment_index_bert', 'R_t+1']

    # 2a. 필요한 컬럼만 추출하고 NaN이 있는 행 제거
    df_clean_chi2 = df_analysis[REQUIRED_COLS_CHI2].dropna()
    df_clean_chi2.columns = ['SI_t', 'R_t+1']

    original_rows = len(df_analysis)
    cleaned_rows = len(df_clean_chi2)

    print(f"원본 데이터 행 수: {original_rows} 행")
    print(f"NaN 제거 후 행 수: {cleaned_rows} 행")

    if cleaned_rows == 0:
        print("경고: NaN 제거 후 데이터가 남아있지 않아 분석을 진행할 수 없습니다.")
        exit()

    # 3. 카이제곱 검정을 위한 범주화

    # 3a. R_t+1 범주화: 상승 vs. 하락/보합
    df_clean_chi2['Return_Category'] = df_clean_chi2['R_t+1'].apply(lambda x: 'Up (상승)' if x > 0 else 'Down/Flat (하락/보합)')

    # 3b. SI_t 범주화 (중앙값 기준)
    sentiment_median = df_clean_chi2['SI_t'].median()
    df_clean_chi2['Sentiment_Category'] = df_clean_chi2['SI_t'].apply(lambda x: 'High Sentiment (긍정)' if x > sentiment_median else 'Low Sentiment (부정)')

    # 4. 교차표 생성 (Cross-Tabulation)
    cross_tab = pd.crosstab(df_clean_chi2['Sentiment_Category'], df_clean_chi2['Return_Category'])

    print("\n--- 3. 감성 지수 (중앙값) vs. 다음 날 주가 변동 교차표 ---")
    print(cross_tab)

    # 5. 카이제곱 검정 (Chi-Squared Test)
    chi2, p, dof, expected = chi2_contingency(cross_tab)

    print("\n--- 4. 카이제곱 검정 결과 ---")
    print(f"카이제곱 통계량 (Chi2 Statistic): {chi2:.4f}")
    print(f"유의 확률 (P-value): {p:.4f}")
    print(f"자유도 (Degrees of Freedom): {dof}")

    print("\n---------------------------------------------------------")
    if p < 0.05:
        print("결론: 감성 지수 범주와 다음 날 주가 변동 범주 사이에는 **통계적으로 유의한 연관성**이 존재합니다.")
    else:
        print("결론: 두 범주 사이에는 **통계적으로 유의한 연관성이 없다**고 볼 수 있습니다.")
    print("---------------------------------------------------------")

except FileNotFoundError:
    print(f"\n[FATAL] 파일을 찾을 수 없습니다. 경로를 확인하세요: {ANALYSIS_PATH}")
except Exception as e:
    print(f"\n[ERROR] 교차분석 중 오류 발생: {e}")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os
from google.colab import drive

warnings.filterwarnings('ignore')

# 1. 파일 로드 및 df_clean_chi2_comp 생성
# Drive 마운트 및 경로 확인이 완료되었다고 가정합니다.

FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    df_analysis = pd.read_csv(FILE_PATH)
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# 분석에 필요한 컬럼 정의 (당일 수익률 R_t와 세 가지 감성 점수)
REQUIRED_COLS_CHI2_COMP = [
    'KOSPI_등락률',
    'avg_pos_score',      # Pos Score
    'avg_neg_score',      # Neg Score
    'avg_neu_score'       # Neu Score
]

# 필요한 컬럼만 추출하여 NaN이 있는 행 전체를 제거
df_clean_comp = df_analysis[REQUIRED_COLS_CHI2_COMP].dropna()
df_clean_comp.columns = ['R_t', 'Pos_Score_t', 'Neg_Score_t', 'Neu_Score_t']

print(f"카이제곱 분석용 'df_clean_comp' 데이터셋 준비 완료. 행 수: {len(df_clean_comp)}")
# R_t 방향 정의 (0: 하락/보합, 1: 상승)
df_clean_comp['R_Direction'] = (df_clean_comp['R_t'] > 0).astype(int)

# 2. 개별 감성 점수에 대한 카이제곱 검정 실행 함수
def run_chi2_analysis(data, score_col, score_name):
    # 감성 점수 3분위수 (Tertile) 생성
    data[f'{score_col}_Tertile'] = pd.cut(
        data[score_col],
        bins=data[score_col].quantile([0, 1/3, 2/3, 1.0]),
        labels=['Low', 'Mid', 'High'],
        include_lowest=True,
        duplicates='drop'
    )

    # 교차표 (Contingency Table) 생성
    crosstab_table = pd.crosstab(data[f'{score_col}_Tertile'], data['R_Direction'],
                                 rownames=[f'{score_name} 강도 (3분위수)'],
                                 colnames=['R_t 등락 (0:하락/보합, 1:상승)'])

    # 카이제곱 (Chi-square) 검정 실행
    chi2, p_value, dof, expected = chi2_contingency(crosstab_table)

    # 결과 요약
    result = {
        'Variable': score_name,
        'Chi2': chi2,
        'P_value': p_value,
        'DOF': dof,
        'Crosstab': crosstab_table,
        'Significance': '유의 (P < 0.05)' if p_value < 0.05 else '비유의 (P >= 0.05)'
    }
    return result

# 3. 세 가지 개별 분석 실행
results = []
print("\n=======================================================")
print("✅ 개별 감성 점수 vs. R_t 등락 방향 카이제곱 검정")
print("=======================================================")

# Model 1: 긍정 점수
pos_result = run_chi2_analysis(df_clean_comp, 'Pos_Score_t', 'Pos_Score')
results.append(pos_result)
print(f"--- 1. Pos_Score (긍정 점수) 검정 결과 ---")
print(pos_result['Crosstab'])
print(f"Chi2: {pos_result['Chi2']:.4f}, P-value: {pos_result['P_value']:.5f}, {pos_result['Significance']}")
print("-" * 40)

# Model 2: 부정 점수
neg_result = run_chi2_analysis(df_clean_comp, 'Neg_Score_t', 'Neg_Score')
results.append(neg_result)
print(f"--- 2. Neg_Score (부정 점수) 검정 결과 ---")
print(neg_result['Crosstab'])
print(f"Chi2: {neg_result['Chi2']:.4f}, P-value: {neg_result['P_value']:.5f}, {neg_result['Significance']}")
print("-" * 40)

# Model 3: 중립 점수
neu_result = run_chi2_analysis(df_clean_comp, 'Neu_Score_t', 'Neu_Score')
results.append(neu_result)
print(f"--- 3. Neu_Score (중립 점수) 검정 결과 ---")
print(neu_result['Crosstab'])
print(f"Chi2: {neu_result['Chi2']:.4f}, P-value: {neu_result['P_value']:.5f}, {neu_result['Significance']}")
print("=======================================================")

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 1. 모든 카이제곱 검정 결과 데이터 수집 (이전에 실행된 결과값 사용)
#
# A. SI (중앙값) vs. R_t+1 방향 (이전 대화 결과)
# 카이제곱 통계량: 0.1188, P-value: 0.7303, DOF: 1
#
# B. SI (3분위수) vs. R_t 방향 (대화 초반 결과)
# 카이제곱 통계량: 1.3131, P-value: 0.51864, DOF: 2
#
# C. Polar SI (3분위수) vs. R_t 방향 (이전 대화 결과)
# 카이제곱 통계량: 3.2200, P-value: 0.19989, DOF: 2
#
# D. 개별 감성 점수 vs. R_t 방향 (직전 대화 결과)
# Pos_Score: Chi2: 0.4281, P-value: 0.80732, DOF: 2
# Neg_Score: Chi2: 4.6687, P-value: 0.09687, DOF: 2
# Neu_Score: Chi2: 2.4178, P-value: 0.29852, DOF: 2


summary_data = [
    # A. SI vs R_t+1 (미래 수익률 방향)
    {'독립 변수 (SI 정의)': 'SI (순수 지수)', '범주화 기준': '중앙값 (2범주)',
     '종속 변수': 'R_t+1 (다음 날 등락 방향)', 'Chi2': 0.1188, 'P_value': 0.7303, 'DOF': 1},

    # B. SI vs R_t (당일 수익률 방향)
    {'독립 변수 (SI 정의)': 'SI (순수 지수)', '범주화 기준': '3분위수 (3범주)',
     '종속 변수': 'R_t (당일 등락 방향)', 'Chi2': 1.3131, 'P_value': 0.51864, 'DOF': 2},

    # C. Polar SI vs R_t (당일 수익률 방향)
    {'독립 변수 (SI 정의)': 'Polar SI (Pos-Neg)', '범주화 기준': '3분위수 (3범주)',
     '종속 변수': 'R_t (당일 등락 방향)', 'Chi2': 3.2200, 'P_value': 0.19989, 'DOF': 2},

    # D. 개별 구성 요소 vs R_t (당일 수익률 방향)
    {'독립 변수 (SI 정의)': 'Pos_Score (긍정 점수)', '범주화 기준': '3분위수 (3범주)',
     '종속 변수': 'R_t (당일 등락 방향)', 'Chi2': 0.4281, 'P_value': 0.80732, 'DOF': 2},
    {'독립 변수 (SI 정의)': 'Neg_Score (부정 점수)', '범주화 기준': '3분위수 (3범주)',
     '종속 변수': 'R_t (당일 등락 방향)', 'Chi2': 4.6687, 'P_value': 0.09687, 'DOF': 2},
    {'독립 변수 (SI 정의)': 'Neu_Score (중립 점수)', '범주화 기준': '3분위수 (3범주)',
     '종속 변수': 'R_t (당일 등락 방향)', 'Chi2': 2.4178, 'P_value': 0.29852, 'DOF': 2},
]

# DataFrame 생성
df_summary = pd.DataFrame(summary_data)

# 유의성 열 추가 (P-value < 0.05)
df_summary['유의성 (P<0.05)'] = df_summary['P_value'].apply(lambda p: '유의 (Significant)' if p < 0.05 else '비유의 (Non-Significant)')

# 결과 출력
print("=======================================================")
print("✅ 감성 지수 관련 카이제곱 검정 **종합 결과 요약**")
print("=======================================================")

# 표 형식으로 출력
print(df_summary.to_markdown(index=False, floatfmt=".4f"))

print("\n--- 최종 결론 ---")
print("모든 카이제곱 검정 (SI, Polar SI, Pos/Neg/Neu 점수) 결과,")
print("P-value가 0.05를 초과하여 **KOSPI 등락 방향**과는 **통계적으로 유의미한 연관성(동행성/예측력)**이 없음을 확인했습니다.")
print("=======================================================")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# ===============================
# 1. Data Preparation
# ===============================
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"
df_analysis = pd.read_csv(FILE_PATH)

REQUIRED_COLS_CHI2_COMP = [
    'KOSPI_등락률',
    'avg_pos_score',
    'avg_neg_score',
    'avg_neu_score'
]

df_clean_comp = df_analysis[REQUIRED_COLS_CHI2_COMP].dropna()
df_clean_comp.columns = ['R_t', 'Pos_Score_t', 'Neg_Score_t', 'Neu_Score_t']
df_clean_comp['R_Direction'] = (df_clean_comp['R_t'] > 0).astype(int)

print(f"Visualization dataset ready. Rows: {len(df_clean_comp)}")

# ===============================
# 2. Helper Function
# ===============================
def prepare_plot_data_and_pvalue(data, score_col):
    # Tertile grouping
    data[f'{score_col}_Tertile'] = pd.cut(
        data[score_col],
        bins=data[score_col].quantile([0, 1/3, 2/3, 1.0]),
        labels=['Low', 'Mid', 'High'],
        include_lowest=True,
        duplicates='drop'
    )

    # Contingency table for Chi-square
    contingency = pd.crosstab(
        data[f'{score_col}_Tertile'],
        data['R_Direction']
    )

    chi2, p_value, _, _ = chi2_contingency(contingency)

    # Upward ratio
    plot_data = contingency.div(contingency.sum(axis=1), axis=0)
    plot_data['Upward Ratio'] = plot_data[1] * 100

    return plot_data[['Upward Ratio']], p_value

# ===============================
# 3. Plotting
# ===============================
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
fig.suptitle('KOSPI Upward Movement Ratio by Sentiment Score Tertiles', fontsize=16)

score_info = [
    ('Pos_Score_t', 'Positive Score (Pos_Score)', 'viridis'),
    ('Neg_Score_t', 'Negative Score (Neg_Score)', 'plasma'),
    ('Neu_Score_t', 'Neutral Score (Neu_Score)', 'cividis')
]

for ax, (score_col, title, palette) in zip(axes, score_info):
    plot_data, p_val = prepare_plot_data_and_pvalue(df_clean_comp.copy(), score_col)

    sns.barplot(
        x=plot_data.index,
        y='Upward Ratio',
        data=plot_data,
        ax=ax,
        palette=palette
    )

    # ----- Value labels on bars -----
    for i, val in enumerate(plot_data['Upward Ratio']):
        ax.text(
            i, val + 0.6,
            f"{val:.1f}%",
            ha='center',
            va='bottom',
            fontsize=10
        )

    # ----- P-value annotation -----
    ax.text(
        0.02, 0.95,
        f"$\chi^2$ test p-value = {p_val:.3f}",
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
    )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Sentiment Intensity')
    ax.set_ylim(40, 65)

axes[0].set_ylabel('KOSPI Upward Ratio (%)')

plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.show()

print("\n--- Visualization with values and p-values complete ---")

# 년도별 및 분기별 $\text{Chi}^2$ 검정 $\text{P}$-$\text{value}$ 시각화 -  Neg_Score

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# 1. 파일 로드 및 데이터 준비 (날짜 컬럼: 'date' 사용)
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    df_analysis = pd.read_csv(FILE_PATH)
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# REQUIRED_COLS에 'date' 사용
REQUIRED_COLS = ['date', 'KOSPI_등락률', 'avg_neg_score']
df_clean_time = df_analysis[REQUIRED_COLS].dropna()

# 'date' 컬럼을 datetime 형식으로 변환하고 인덱스로 설정
df_clean_time['date'] = pd.to_datetime(df_clean_time['date'])
df_clean_time = df_clean_time.set_index('date')

# 컬럼명 정리 및 R_t 방향 정의
df_clean_time.columns = ['R_t', 'Neg_Score_t']
df_clean_time['R_Direction'] = (df_clean_time['R_t'] > 0).astype(int) # 0:Down/Flat, 1:Up

# Neg_Score를 3분위수로 범주화
df_clean_time['Neg_Tertile'] = pd.cut(
    df_clean_time['Neg_Score_t'],
    bins=df_clean_time['Neg_Score_t'].quantile([0, 1/3, 2/3, 1.0]),
    labels=['Low', 'Mid', 'High'],
    include_lowest=True,
    duplicates='drop'
)
print(f"시간 분할 분석용 데이터셋 준비 완료. 행 수: {len(df_clean_time)}")


# 2. 기간별 Chi-squared Test 실행 함수
def run_chi2_by_period(data, period_name):
    # 데이터가 0개이거나 범주가 부족한 경우 처리
    if len(data) < 30 or data['Neg_Tertile'].nunique() < 2 or data['R_Direction'].nunique() < 2:
        return {'Period': period_name, 'Count': len(data), 'Chi2': np.nan, 'P_value': np.nan, 'Conclusion': 'Data Insufficient'}

    crosstab_table = pd.crosstab(data['Neg_Tertile'], data['R_Direction'])

    # 카이제곱 검정
    chi2, p_value, dof, expected = chi2_contingency(crosstab_table)

    conclusion = '유의 (P < 0.05)' if p_value < 0.05 else '비유의'

    return {'Period': period_name, 'Count': len(data), 'Chi2': chi2, 'P_value': p_value, 'Conclusion': conclusion}


# 3. 년도별 분할 분석 실행
print("\n=======================================================")
print("✅ 년도별 Neg_Score vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
yearly_results = []
for year, df_year in df_clean_time.groupby(df_clean_time.index.year):
    result = run_chi2_by_period(df_year, str(year))
    yearly_results.append(result)

df_yearly = pd.DataFrame(yearly_results)
print(df_yearly.to_markdown(index=False, floatfmt=".4f"))
print("-" * 55)


# 4. 분기별 분할 분석 실행
print("\n=======================================================")
print("✅ 분기별 Neg_Score vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
quarterly_results = []
for name, df_quarter in df_clean_time.groupby([df_clean_time.index.year, df_clean_time.index.quarter]):
    period_name = f"{name[0]}-Q{name[1]}"
    result = run_chi2_by_period(df_quarter, period_name)
    quarterly_results.append(result)

df_quarterly = pd.DataFrame(quarterly_results)
print(df_quarterly.to_markdown(index=False, floatfmt=".4f"))
print("=======================================================")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 분석 결과를 DataFrame으로 준비 (입력받은 결과 사용)
yearly_data = {
    'Period': [2020, 2021, 2022, 2023, 2024, 2025],
    'P_value': [0.0777, 0.0372, 0.8764, 0.1396, 0.5102, 0.8612]
}
df_yearly = pd.DataFrame(yearly_data)
df_yearly['Period'] = df_yearly['Period'].astype(str)

quarterly_data = {
    'Period': ['2020-Q1', '2020-Q2', '2020-Q3', '2020-Q4',
               '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4',
               '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4',
               '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4',
               '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4',
               '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4'],
    'P_value': [0.7889, 0.6761, 0.2062, 0.5211,
                0.0223, 0.4922, 0.8961, 0.0765,
                0.8763, 0.3258, 0.9996, 0.5037,
                0.7645, 0.1071, 0.6448, 0.1946,
                0.5024, 0.3057, 0.6893, 0.3763,
                0.3525, 0.8873, 0.4642, 0.1152]
}
df_quarterly = pd.DataFrame(quarterly_data)

# 2. 그래프 설정
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 100

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('Time-Varying Association between Negative Score and KOSPI Direction (P-values)', fontsize=18, fontweight='bold')

# --- 2-1. 년도별 P-value 시각화 ---
axes[0].plot(df_yearly['Period'], df_yearly['P_value'], marker='o', linestyle='-', color='darkblue')
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[0].set_title('Annual Chi-squared P-values', fontsize=14)
axes[0].set_ylabel('P-value')
axes[0].set_xlabel('Year')
axes[0].set_ylim(0, 1.0)
axes[0].legend()

# 년도별 수치 표기 추가 (fontsize=11)
for i, row in df_yearly.iterrows():
    color = 'red' if row['P_value'] < 0.05 else 'black'
    font_weight = 'bold' if row['P_value'] < 0.05 else 'normal'
    axes[0].text(row['Period'], row['P_value'] + 0.04, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=11, color=color, fontweight=font_weight)


# --- 2-2. 분기별 P-value 시각화 ---
x_labels = df_quarterly['Period']
x_ticks = np.arange(len(x_labels))

axes[1].plot(x_ticks, df_quarterly['P_value'], marker='o', linestyle='-', color='darkgreen')
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[1].set_title('Quarterly Chi-squared P-values', fontsize=14)
axes[1].set_ylabel('P-value')
axes[1].set_xlabel('Period (Quarter)')
axes[1].set_ylim(0, 1.0)
axes[1].set_xticks(x_ticks[::2])
axes[1].set_xticklabels(x_labels[::2], rotation=45, ha='right')
axes[1].legend()

# 분기별 수치 표기 추가 (fontsize=10)
for i, row in df_quarterly.iterrows():
    # P-value가 0.7 이상인 경우 텍스트를 아래로, 나머지는 위로 배치하여 겹침 방지
    offset = -0.06 if row['P_value'] > 0.7 else 0.04
    color = 'red' if row['P_value'] < 0.05 else 'black'
    font_weight = 'bold' if row['P_value'] < 0.05 else 'normal'
    axes[1].text(i, row['P_value'] + offset, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=10, color=color, fontweight=font_weight)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

print("\n--- 시각화 완료 (수치 표기 크기 확대) ---")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# 1. 파일 로드 및 데이터 준비 (날짜 컬럼: 'date' 사용)
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    df_analysis = pd.read_csv(FILE_PATH)
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# REQUIRED_COLS에 'avg_pos_score' 사용
REQUIRED_COLS = ['date', 'KOSPI_등락률', 'avg_pos_score']
# 'avg_neg_score' 대신 'avg_pos_score'를 사용하여 데이터프레임 로드 및 결측치 제거
df_clean_time = df_analysis[REQUIRED_COLS].dropna()

# 'date' 컬럼을 datetime 형식으로 변환하고 인덱스로 설정
df_clean_time['date'] = pd.to_datetime(df_clean_time['date'])
df_clean_time = df_clean_time.set_index('date')

# 컬럼명 정리 및 R_t 방향 정의
df_clean_time.columns = ['R_t', 'Pos_Score_t']  # 컬럼명을 Pos_Score_t로 변경
df_clean_time['R_Direction'] = (df_clean_time['R_t'] > 0).astype(int) # 0:Down/Flat, 1:Up

# Pos_Score를 3분위수로 범주화
df_clean_time['Pos_Tertile'] = pd.cut(
    df_clean_time['Pos_Score_t'],
    bins=df_clean_time['Pos_Score_t'].quantile([0, 1/3, 2/3, 1.0]),
    labels=['Low', 'Mid', 'High'],
    include_lowest=True,
    duplicates='drop'
)
print(f"시간 분할 분석용 데이터셋 준비 완료. 행 수: {len(df_clean_time)}")


# 2. 기간별 Chi-squared Test 실행 함수
def run_chi2_by_period_pos(data, period_name):
    # 데이터가 0개이거나 범주가 부족한 경우 처리
    # 'Neg_Tertile' 대신 'Pos_Tertile' 사용
    if len(data) < 30 or data['Pos_Tertile'].nunique() < 2 or data['R_Direction'].nunique() < 2:
        return {'Period': period_name, 'Count': len(data), 'Chi2': np.nan, 'P_value': np.nan, 'Conclusion': 'Data Insufficient'}

    # 'Neg_Tertile' 대신 'Pos_Tertile' 사용
    crosstab_table = pd.crosstab(data['Pos_Tertile'], data['R_Direction'])

    # 카이제곱 검정
    chi2, p_value, dof, expected = chi2_contingency(crosstab_table)

    conclusion = '유의 (P < 0.05)' if p_value < 0.05 else '비유의'

    return {'Period': period_name, 'Count': len(data), 'Chi2': chi2, 'P_value': p_value, 'Conclusion': conclusion}


# 3. 년도별 분할 분석 실행
print("\n=======================================================")
print("✅ 년도별 Pos_Score vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
yearly_results = []
for year, df_year in df_clean_time.groupby(df_clean_time.index.year):
    # 함수 이름 변경
    result = run_chi2_by_period_pos(df_year, str(year))
    yearly_results.append(result)

df_yearly = pd.DataFrame(yearly_results)
print(df_yearly.to_markdown(index=False, floatfmt=".4f"))
print("-" * 55)


# 4. 분기별 분할 분석 실행
print("\n=======================================================")
print("✅ 분기별 Pos_Score vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
quarterly_results = []
for name, df_quarter in df_clean_time.groupby([df_clean_time.index.year, df_clean_time.index.quarter]):
    period_name = f"{name[0]}-Q{name[1]}"
    # 함수 이름 변경
    result = run_chi2_by_period_pos(df_quarter, period_name)
    quarterly_results.append(result)

df_quarterly = pd.DataFrame(quarterly_results)
print(df_quarterly.to_markdown(index=False, floatfmt=".4f"))
print("=======================================================")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Prepare DataFrames from the provided Pos_Score results
# =================================================================
# Annual Pos_Score vs. R_t Direction Chi-squared P-values
yearly_data_pos = {
    'Period': [2020, 2021, 2022, 2023, 2024, 2025],
    'P_value': [0.6180, 0.9779, 0.5643, 0.5353, 0.1292, 0.3659]
}
df_yearly_pos = pd.DataFrame(yearly_data_pos)
df_yearly_pos['Period'] = df_yearly_pos['Period'].astype(str)

# Quarterly Pos_Score vs. R_t Direction Chi-squared P-values
quarterly_data_pos = {
    'Period': ['2020-Q1', '2020-Q2', '2020-Q3', '2020-Q4',
               '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4',
               '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4',
               '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4',
               '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4',
               '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4'],
    'P_value': [0.2431, 0.3843, 0.1618, 0.8873,
                0.3033, 0.9916, 0.8375, 0.3232,
                0.2888, 0.9471, 0.5398, 0.4476,
                0.8229, 0.0998, 0.4153, 0.3606,
                0.4096, 0.3670, 0.8980, 0.5985,
                0.5758, 0.5613, 0.5599, 0.7066]
}
df_quarterly_pos = pd.DataFrame(quarterly_data_pos)
# =================================================================

# 2. Plotting Setup (Using English standard settings)
sns.set_style("whitegrid")
# Ensure standard font settings are used (no explicit Korean font)
plt.rcParams['figure.dpi'] = 100

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('Time-Varying Association between Positive Score and KOSPI Direction (P-values)', fontsize=18, fontweight='bold')

# --- 2-1. Annual P-value Visualization ---
axes[0].plot(df_yearly_pos['Period'], df_yearly_pos['P_value'], marker='o', linestyle='-', color='darkgreen')
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[0].set_title('Annual Chi-squared P-values', fontsize=14)
axes[0].set_ylabel('P-value')
axes[0].set_xlabel('Year')
axes[0].set_ylim(0, 1.0)
axes[0].legend()

# Add P-value labels (fontsize=11)
for i, row in df_yearly_pos.iterrows():
    color = 'red' if row['P_value'] < 0.05 else 'black'
    font_weight = 'bold' if row['P_value'] < 0.05 else 'normal'
    axes[0].text(row['Period'], row['P_value'] + 0.04, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=11, color=color, fontweight=font_weight)


# --- 2-2. Quarterly P-value Visualization ---
x_labels = df_quarterly_pos['Period']
x_ticks = np.arange(len(x_labels))

axes[1].plot(x_ticks, df_quarterly_pos['P_value'], marker='o', linestyle='-', color='darkblue')
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[1].set_title('Quarterly Chi-squared P-values', fontsize=14)
axes[1].set_ylabel('P-value')
axes[1].set_xlabel('Period (Quarter)')
axes[1].set_ylim(0, 1.0)
axes[1].set_xticks(x_ticks[::2]) # Display ticks every 2 periods
axes[1].set_xticklabels(x_labels[::2], rotation=45, ha='right')
axes[1].legend()

# Add P-value labels (fontsize=10)
for i, row in df_quarterly_pos.iterrows():
    # Adjust offset to prevent label overlap for high P-values
    offset = -0.06 if row['P_value'] > 0.7 else 0.04
    color = 'red' if row['P_value'] < 0.05 else 'black'
    font_weight = 'bold' if row['P_value'] < 0.05 else 'normal'
    axes[1].text(i, row['P_value'] + offset, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=10, color=color, fontweight=font_weight)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

print("\n--- Positive Score (Pos Score) P-value Visualization Completed ---")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# 1. 파일 로드 및 데이터 준비 (날짜 컬럼: 'date' 사용)
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    df_analysis = pd.read_csv(FILE_PATH)
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# REQUIRED_COLS에 'avg_neu_score' 사용
REQUIRED_COLS = ['date', 'KOSPI_등락률', 'avg_neu_score']
# 'avg_neu_score'를 사용하여 데이터프레임 로드 및 결측치 제거
df_clean_time = df_analysis[REQUIRED_COLS].dropna()

# 'date' 컬럼을 datetime 형식으로 변환하고 인덱스로 설정
df_clean_time['date'] = pd.to_datetime(df_clean_time['date'])
df_clean_time = df_clean_time.set_index('date')

# 컬럼명 정리 및 R_t 방향 정의
df_clean_time.columns = ['R_t', 'Neu_Score_t']  # 컬럼명을 Neu_Score_t로 변경
df_clean_time['R_Direction'] = (df_clean_time['R_t'] > 0).astype(int) # 0:Down/Flat, 1:Up

# Neu_Score를 3분위수로 범주화
df_clean_time['Neu_Tertile'] = pd.cut(
    df_clean_time['Neu_Score_t'],
    bins=df_clean_time['Neu_Score_t'].quantile([0, 1/3, 2/3, 1.0]),
    labels=['Low', 'Mid', 'High'],
    include_lowest=True,
    duplicates='drop'
)
print(f"시간 분할 분석용 데이터셋 준비 완료. 행 수: {len(df_clean_time)}")


# 2. 기간별 Chi-squared Test 실행 함수
def run_chi2_by_period_neu(data, period_name):
    # 데이터가 0개이거나 범주가 부족한 경우 처리
    if len(data) < 30 or data['Neu_Tertile'].nunique() < 2 or data['R_Direction'].nunique() < 2:
        return {'Period': period_name, 'Count': len(data), 'Chi2': np.nan, 'P_value': np.nan, 'Conclusion': 'Data Insufficient'}

    # 'Neu_Tertile' 사용
    crosstab_table = pd.crosstab(data['Neu_Tertile'], data['R_Direction'])

    # 카이제곱 검정
    chi2, p_value, dof, expected = chi2_contingency(crosstab_table)

    conclusion = '유의 (P < 0.05)' if p_value < 0.05 else '비유의'

    return {'Period': period_name, 'Count': len(data), 'Chi2': chi2, 'P_value': p_value, 'Conclusion': conclusion}


# 3. 년도별 분할 분석 실행
print("\n=======================================================")
print("✅ 년도별 Neu_Score vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
yearly_results = []
for year, df_year in df_clean_time.groupby(df_clean_time.index.year):
    result = run_chi2_by_period_neu(df_year, str(year))
    yearly_results.append(result)

df_yearly = pd.DataFrame(yearly_results)
print(df_yearly.to_markdown(index=False, floatfmt=".4f"))
print("-" * 55)


# 4. 분기별 분할 분석 실행
print("\n=======================================================")
print("✅ 분기별 Neu_Score vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
quarterly_results = []
for name, df_quarter in df_clean_time.groupby([df_clean_time.index.year, df_clean_time.index.quarter]):
    period_name = f"{name[0]}-Q{name[1]}"
    result = run_chi2_by_period_neu(df_quarter, period_name)
    quarterly_results.append(result)

df_quarterly = pd.DataFrame(quarterly_results)
print(df_quarterly.to_markdown(index=False, floatfmt=".4f"))
print("=======================================================")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Prepare DataFrames from the provided Neu_Score results
# =================================================================
# Annual Neu_Score vs. R_t Direction Chi-squared P-values
yearly_data_neu = {
    'Period': [2020, 2021, 2022, 2023, 2024, 2025],
    'P_value': [0.8271, 0.7427, 0.1989, 0.1783, 0.0202, 0.6399]
}
df_yearly_neu = pd.DataFrame(yearly_data_neu)
df_yearly_neu['Period'] = df_yearly_neu['Period'].astype(str)

# Quarterly Neu_Score vs. R_t Direction Chi-squared P-values
quarterly_data_neu = {
    'Period': ['2020-Q1', '2020-Q2', '2020-Q3', '2020-Q4',
               '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4',
               '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4',
               '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4',
               '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4',
               '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4'],
    'P_value': [0.2466, 0.7555, 0.6196, 0.9567,
                0.0964, 0.4124, 0.8940, 0.7989,
                0.2592, 0.2199, 0.8397, 0.2835,
                0.8508, 0.0234, 0.2885, 0.5258,
                0.4224, 0.7915, 0.0144, 0.3512,
                0.4206, 0.6776, 0.2994, 0.5633]
}
df_quarterly_neu = pd.DataFrame(quarterly_data_neu)
# =================================================================

# 2. Plotting Setup
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('Time-Varying Association between Neutral Score and KOSPI Direction (P-values)', fontsize=18, fontweight='bold')

# --- 2-1. Annual P-value Visualization ---
axes[0].plot(df_yearly_neu['Period'], df_yearly_neu['P_value'], marker='o', linestyle='-', color='darkred')
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[0].set_title('Annual Chi-squared P-values', fontsize=14)
axes[0].set_ylabel('P-value')
axes[0].set_xlabel('Year')
axes[0].set_ylim(0, 1.0)
axes[0].legend()

# Add P-value labels (fontsize=11)
for i, row in df_yearly_neu.iterrows():
    color = 'red' if row['P_value'] < 0.05 else 'black'
    font_weight = 'bold' if row['P_value'] < 0.05 else 'normal'
    axes[0].text(row['Period'], row['P_value'] + 0.04, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=11, color=color, fontweight=font_weight)


# --- 2-2. Quarterly P-value Visualization ---
x_labels = df_quarterly_neu['Period']
x_ticks = np.arange(len(x_labels))

axes[1].plot(x_ticks, df_quarterly_neu['P_value'], marker='o', linestyle='-', color='darkorange')
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[1].set_title('Quarterly Chi-squared P-values', fontsize=14)
axes[1].set_ylabel('P-value')
axes[1].set_xlabel('Period (Quarter)')
axes[1].set_ylim(0, 1.0)
axes[1].set_xticks(x_ticks[::2]) # Display ticks every 2 periods
axes[1].set_xticklabels(x_labels[::2], rotation=45, ha='right')
axes[1].legend()

# Add P-value labels (fontsize=10)
for i, row in df_quarterly_neu.iterrows():
    # Highlight significant P-values and adjust offset for readability
    is_significant = row['P_value'] < 0.05
    offset = -0.06 if row['P_value'] > 0.7 or is_significant else 0.04
    color = 'red' if is_significant else 'black'
    font_weight = 'bold' if is_significant else 'normal'

    axes[1].text(i, row['P_value'] + offset, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=10, color=color, fontweight=font_weight)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

print("\n--- Neutral Score (Neu Score) P-value Visualization Completed ---")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# 1. 파일 로드 및 데이터 준비
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    df_analysis = pd.read_csv(FILE_PATH)
    # 세 가지 감성 점수와 KOSPI 등락률을 모두 로드합니다.
    REQUIRED_COLS = ['date', 'KOSPI_등락률', 'avg_neg_score', 'avg_pos_score', 'avg_neu_score']
    df_integrated = df_analysis[REQUIRED_COLS].dropna()
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# 'date' 컬럼 처리 및 R_t 방향 정의
df_integrated['date'] = pd.to_datetime(df_integrated['date'])
df_integrated = df_integrated.set_index('date')
df_integrated['R_Direction'] = (df_integrated['KOSPI_등락률'] > 0).astype(int).replace({0: 'Down/Flat', 1: 'Up'})


# 2. 통합 감성 점수 유형 (Score_Type) 정의
# 세 점수 중 가장 높은 값을 갖는 감성을 해당 일의 대표 감성 유형으로 지정
def get_max_score_type(row):
    scores = {
        'Neg': row['avg_neg_score'],
        'Pos': row['avg_pos_score'],
        'Neu': row['avg_neu_score']
    }
    # 값이 동일한 경우, pandas의 idxmax는 가장 먼저 나오는 컬럼명을 반환합니다.
    return max(scores, key=scores.get)

# Score_Type 컬럼 생성
df_integrated['Score_Type'] = df_integrated.apply(get_max_score_type, axis=1)

print(f"시간 분할 분석용 통합 데이터셋 준비 완료. 행 수: {len(df_integrated)}")
print("\nScore_Type 분포:")
print(df_integrated['Score_Type'].value_counts().to_markdown())


# 3. 기간별 통합 Chi-squared Test 실행 함수
def run_chi2_integrated(data, period_name):

    # 데이터 부족/범주 부족 시 처리
    if len(data) < 30 or data['Score_Type'].nunique() < 2 or data['R_Direction'].nunique() < 2:
        return {'Period': period_name, 'Count': len(data), 'Chi2': np.nan, 'P_value': np.nan, 'Conclusion': 'Data Insufficient'}

    # Score_Type vs. R_Direction 교차표 생성
    crosstab_table = pd.crosstab(data['Score_Type'], data['R_Direction'])

    # 카이제곱 검정
    chi2, p_value, dof, expected = chi2_contingency(crosstab_table)

    conclusion = '유의 (P < 0.05)' if p_value < 0.05 else '비유의'

    # 잔차 분석을 위해 crosstab과 residuals도 함께 반환 (유의할 경우 잔차 분석을 바로 할 수 있도록)
    residuals = (crosstab_table - expected) / np.sqrt(expected)

    return {'Period': period_name, 'Count': len(data), 'Chi2': chi2, 'P_value': p_value, 'Conclusion': conclusion, 'Residuals': residuals}


# 4. 년도별 통합 분석 실행
print("\n=======================================================")
print("✅ 년도별 Score_Type vs. R_t 방향 Chi-squared 검정 (통합)")
print("=======================================================")
yearly_results_integrated = []
for year, df_year in df_integrated.groupby(df_integrated.index.year):
    result = run_chi2_integrated(df_year, str(year))
    yearly_results_integrated.append(result)

df_yearly_integrated = pd.DataFrame(yearly_results_integrated)
# Residuals 컬럼 제거 후 출력
print(df_yearly_integrated[['Period', 'Count', 'Chi2', 'P_value', 'Conclusion']].to_markdown(index=False, floatfmt=".4f"))
print("-" * 55)


# 5. 분기별 통합 분석 실행
print("\n=======================================================")
print("✅ 분기별 Score_Type vs. R_t 방향 Chi-squared 검정 (통합)")
print("=======================================================")
quarterly_results_integrated = []
for name, df_quarter in df_integrated.groupby([df_integrated.index.year, df_integrated.index.quarter]):
    period_name = f"{name[0]}-Q{name[1]}"
    result = run_chi2_integrated(df_quarter, period_name)
    quarterly_results_integrated.append(result)

df_quarterly_integrated = pd.DataFrame(quarterly_results_integrated)
# Residuals 컬럼 제거 후 출력
print(df_quarterly_integrated[['Period', 'Count', 'Chi2', 'P_value', 'Conclusion']].to_markdown(index=False, floatfmt=".4f"))
print("=======================================================")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Prepare DataFrames from the provided Integrated Score_Type results
# =================================================================
# Annual Score_Type vs. R_t Direction Chi-squared P-values
yearly_data_int = {
    'Period': [2020, 2021, 2022, 2023, 2024, 2025],
    'P_value': [0.8603, 0.8365, 0.6151, 0.4087, 0.0332, 0.5223]
}
df_yearly_int = pd.DataFrame(yearly_data_int)
df_yearly_int['Period'] = df_yearly_int['Period'].astype(str)

# Quarterly Score_Type vs. R_t Direction Chi-squared P-values
quarterly_data_int = {
    'Period': ['2020-Q1', '2020-Q2', '2020-Q3', '2020-Q4',
               '2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4',
               '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4',
               '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4',
               '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4',
               '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4'],
    'P_value': [0.7071, 0.8250, 0.7378, 1.0000,
                0.4179, 1.0000, 0.8959, 0.8772,
                1.0000, 0.3327, 0.4247, 0.3550,
                1.0000, 0.0506, 0.1964, 0.0369,
                0.5338, 0.2833, 0.0109, 0.9489,
                0.8109, 0.5567, 0.3056, 1.0000]
}
df_quarterly_int = pd.DataFrame(quarterly_data_int)
# =================================================================

# 2. Plotting Setup
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('Time-Varying Association between Dominant Score Type and KOSPI Direction (P-values)', fontsize=18, fontweight='bold')

# --- 2-1. Annual P-value Visualization ---
axes[0].plot(df_yearly_int['Period'], df_yearly_int['P_value'], marker='o', linestyle='-', color='purple')
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[0].set_title('Annual Chi-squared P-values (Dominant Score Type)', fontsize=14)
axes[0].set_ylabel('P-value')
axes[0].set_xlabel('Year')
axes[0].set_ylim(0, 1.05) # Max 1.05 for 1.00 P-value label placement
axes[0].legend()

# Add P-value labels (fontsize=11)
for i, row in df_yearly_int.iterrows():
    color = 'red' if row['P_value'] < 0.05 else 'black'
    font_weight = 'bold' if row['P_value'] < 0.05 else 'normal'
    axes[0].text(row['Period'], row['P_value'] + 0.04, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=11, color=color, fontweight=font_weight)


# --- 2-2. Quarterly P-value Visualization ---
x_labels = df_quarterly_int['Period']
x_ticks = np.arange(len(x_labels))

axes[1].plot(x_ticks, df_quarterly_int['P_value'], marker='o', linestyle='-', color='darkcyan')
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Significance Level (0.05)')
axes[1].set_title('Quarterly Chi-squared P-values (Dominant Score Type)', fontsize=14)
axes[1].set_ylabel('P-value')
axes[1].set_xlabel('Period (Quarter)')
axes[1].set_ylim(0, 1.05)
axes[1].set_xticks(x_ticks[::2]) # Display ticks every 2 periods
axes[1].set_xticklabels(x_labels[::2], rotation=45, ha='right')
axes[1].legend()

# Add P-value labels (fontsize=10)
for i, row in df_quarterly_int.iterrows():
    # Highlight significant P-values and adjust offset for readability (esp for 1.00)
    is_significant = row['P_value'] < 0.05
    offset = -0.06 if row['P_value'] > 0.9 or is_significant else 0.04
    color = 'red' if is_significant else 'black'
    font_weight = 'bold' if is_significant else 'normal'

    axes[1].text(i, row['P_value'] + offset, f"{row['P_value']:.4f}",
                 ha='center', va='bottom', fontsize=10, color=color, fontweight=font_weight)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

print("\n--- Dominant Score Type P-value Visualization Completed ---")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# 1. 파일 로드 및 데이터 준비
FILE_PATH = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    df_analysis = pd.read_csv(FILE_PATH)
except Exception as e:
    print(f"오류: 파일을 로드하지 못했습니다. 경로를 확인해 주세요: {e}")
    exit()

# 필요한 컬럼 정의 (date, R_t, Pos, Neg, Neu)
REQUIRED_COLS = ['date', 'KOSPI_등락률', 'avg_pos_score', 'avg_neg_score', 'avg_neu_score']
df_clean_time = df_analysis[REQUIRED_COLS].dropna()

# 'date' 컬럼을 datetime 형식으로 변환하고 인덱스로 설정
df_clean_time['date'] = pd.to_datetime(df_clean_time['date'])
df_clean_time = df_clean_time.set_index('date')

# 컬럼명 정리 및 R_t 방향 정의
df_clean_time.columns = ['R_t', 'Pos_Score_t', 'Neg_Score_t', 'Neu_Score_t']
df_clean_time['R_Direction'] = (df_clean_time['R_t'] > 0).astype(int) # 0:Down/Flat, 1:Up

# 2. 기간별 Chi-squared Test 실행 함수
def run_chi2_by_period(data, score_col, period_name):
    # 점수를 3분위수로 범주화
    data['Tertile'] = pd.cut(
        data[score_col],
        bins=data[score_col].quantile([0, 1/3, 2/3, 1.0]),
        labels=['Low', 'Mid', 'High'],
        include_lowest=True,
        duplicates='drop'
    )

    # 데이터가 0개이거나 범주가 부족한 경우 처리
    if len(data) < 30 or data['Tertile'].nunique() < 2 or data['R_Direction'].nunique() < 2:
        return {'Period': period_name, 'Score_Type': score_col.replace('_Score_t', ''), 'Count': len(data), 'Chi2': np.nan, 'P_value': np.nan, 'Conclusion': 'Data Insufficient'}

    crosstab_table = pd.crosstab(data['Tertile'], data['R_Direction'])

    # 카이제곱 검정
    try:
        chi2, p_value, dof, expected = chi2_contingency(crosstab_table)
    except ValueError:
        return {'Period': period_name, 'Score_Type': score_col.replace('_Score_t', ''), 'Count': len(data), 'Chi2': np.nan, 'P_value': np.nan, 'Conclusion': 'Cell Count Zero'}

    conclusion = '유의 (P < 0.05)' if p_value < 0.05 else '비유의'

    return {'Period': period_name, 'Score_Type': score_col.replace('_Score_t', ''), 'Count': len(data), 'Chi2': chi2, 'P_value': p_value, 'Conclusion': conclusion}


# 3. 년도별 분할 분석 실행 (Pos, Neg, Neu 모두)
print("\n=======================================================")
print("✅ 년도별 Pos/Neg/Neu vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
yearly_results_all = []
score_cols = ['Pos_Score_t', 'Neg_Score_t', 'Neu_Score_t']

for year, df_year in df_clean_time.groupby(df_clean_time.index.year):
    for score_col in score_cols:
        result = run_chi2_by_period(df_year.copy(), score_col, str(year))
        yearly_results_all.append(result)

df_yearly_all = pd.DataFrame(yearly_results_all)
print(df_yearly_all.to_markdown(index=False, floatfmt=".4f"))
print("-" * 70)


# 4. 분기별 분할 분석 실행 (Pos, Neg, Neu 모두)
print("\n=======================================================")
print("✅ 분기별 Pos/Neg/Neu vs. R_t 방향 Chi-squared 검정")
print("=======================================================")
quarterly_results_all = []
for name, df_quarter in df_clean_time.groupby([df_clean_time.index.year, df_clean_time.index.quarter]):
    period_name = f"{name[0]}-Q{name[1]}"
    for score_col in score_cols:
        result = run_chi2_by_period(df_quarter.copy(), score_col, period_name)
        quarterly_results_all.append(result)

df_quarterly_all = pd.DataFrame(quarterly_results_all)
print(df_quarterly_all.to_markdown(index=False, floatfmt=".4f"))
print("=======================================================")

In [ ]:
import matplotlib.pyplot as plt

# Neg_Score에 대한 P-value만 추출하여 시각화
neg_results = df_quarterly_all[df_quarterly_all['Score_Type'] == 'Neg']

plt.figure(figsize=(15, 5))
plt.plot(neg_results['Period'], neg_results['P_value'], marker='o', linestyle='-', color='red', label='P-value (Neg)')
plt.axhline(y=0.05, color='blue', linestyle='--', label='Significance Level (0.05)')
plt.fill_between(neg_results['Period'], 0, 0.05, color='red', alpha=0.1, label='Significance Zone')

plt.title('Quarterly Dynamics of Information Sensitivity (Negative Sentiment)')
plt.xticks(rotation=45)
plt.ylabel('P-value')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# Time-varying Chi-squared Test Visualization (Final)
# =========================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# =========================================================
# 1. Data Preparation
# =========================================================

# ----- Annual data -----
yearly_data = [
    (2020, 'Pos', 0.7728), (2020, 'Neg', 0.4533), (2020, 'Neu', 0.9870),
    (2021, 'Pos', 0.3515), (2021, 'Neg', 0.1454), (2021, 'Neu', 0.8822),
    (2022, 'Pos', 0.0638), (2022, 'Neg', 0.9369), (2022, 'Neu', 0.1868),
    (2023, 'Pos', 0.7684), (2023, 'Neg', 0.2997), (2023, 'Neu', 0.5477),
    (2024, 'Pos', 0.1526), (2024, 'Neg', 0.2862), (2024, 'Neu', 0.0720),
    (2025, 'Pos', 0.9856), (2025, 'Neg', 0.5965), (2025, 'Neu', 0.4968),
]

df_yearly = pd.DataFrame(yearly_data, columns=['Period', 'Score_Type', 'P_value'])
df_yearly['Period'] = df_yearly['Period'].astype(str)

# ----- Quarterly data -----
quarterly_data = [
    ('2020-Q1','Pos',0.0608),('2020-Q1','Neg',0.6369),('2020-Q1','Neu',0.3592),
    ('2020-Q2','Pos',0.5920),('2020-Q2','Neg',0.8600),('2020-Q2','Neu',0.1768),
    ('2020-Q3','Pos',0.1927),('2020-Q3','Neg',0.3247),('2020-Q3','Neu',0.8576),
    ('2020-Q4','Pos',0.9650),('2020-Q4','Neg',0.6060),('2020-Q4','Neu',0.9650),

    ('2021-Q1','Pos',0.6258),('2021-Q1','Neg',0.7650),('2021-Q1','Neu',0.2802),
    ('2021-Q2','Pos',0.4106),('2021-Q2','Neg',0.4106),('2021-Q2','Neu',0.9338),
    ('2021-Q3','Pos',0.5453),('2021-Q3','Neg',0.7363),('2021-Q3','Neu',0.7363),
    ('2021-Q4','Pos',0.2628),('2021-Q4','Neg',1.0000),('2021-Q4','Neu',0.4660),

    ('2022-Q1','Pos',0.2927),('2022-Q1','Neg',0.9823),('2022-Q1','Neu',0.2123),
    ('2022-Q2','Pos',0.4950),('2022-Q2','Neg',0.8405),('2022-Q2','Neu',0.8405),
    ('2022-Q3','Pos',0.8262),('2022-Q3','Neg',0.8262),('2022-Q3','Neu',0.8262),
    ('2022-Q4','Pos',0.0632),('2022-Q4','Neg',0.4593),('2022-Q4','Neu',0.2838),

    ('2023-Q1','Pos',0.9009),('2023-Q1','Neg',0.6910),('2023-Q1','Neu',0.3231),
    ('2023-Q2','Pos',0.0471),('2023-Q2','Neg',0.1192),('2023-Q2','Neu',0.0178),
    ('2023-Q3','Pos',0.3749),('2023-Q3','Neg',0.5037),('2023-Q3','Neu',0.2097),
    ('2023-Q4','Pos',0.2718),('2023-Q4','Neg',0.2718),('2023-Q4','Neu',0.4101),

    ('2024-Q1','Pos',0.7108),('2024-Q1','Neg',0.5363),('2024-Q1','Neu',0.9511),
    ('2024-Q2','Pos',0.0562),('2024-Q2','Neg',0.1254),('2024-Q2','Neu',0.7650),
    ('2024-Q3','Pos',0.8697),('2024-Q3','Neg',0.5352),('2024-Q3','Neu',0.0288),
    ('2024-Q4','Pos',0.8744),('2024-Q4','Neg',0.4136),('2024-Q4','Neu',0.5094),

    ('2025-Q1','Pos',0.5776),('2025-Q1','Neg',0.0050),('2025-Q1','Neu',0.9291),
    ('2025-Q2','Pos',0.3934),('2025-Q2','Neg',0.1340),('2025-Q2','Neu',0.1663),
    ('2025-Q3','Pos',0.3447),('2025-Q3','Neg',0.5612),('2025-Q3','Neu',0.4388),
    ('2025-Q4','Pos',0.4621),('2025-Q4','Neg',0.5105),('2025-Q4','Neu',0.5105),
]

df_quarterly = pd.DataFrame(quarterly_data, columns=['Period','Score_Type','P_value'])

# =========================================================
# 2. Visualization
# =========================================================

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 110

fig, axes = plt.subplots(3, 2, figsize=(20, 18), sharey='row')
fig.suptitle(
    "Time-Varying Association between Sentiment Scores and KOSPI Return Direction\n(Chi-squared Test P-values)",
    fontsize=22, fontweight="bold", y=1.02
)

score_types = ['Pos', 'Neg', 'Neu']
colors = ['blue', 'red', 'purple']

for i, score in enumerate(score_types):

    # ---------- Annual ----------
    df_y = df_yearly[df_yearly['Score_Type'] == score]

    axes[i,0].plot(df_y['Period'], df_y['P_value'], marker='o', color=colors[i])
    axes[i,0].axhline(0.05, linestyle='--', color='black', linewidth=1.3)
    axes[i,0].set_title(f"Annual P-values ({score})", fontsize=15)
    axes[i,0].set_ylabel("P-value")
    axes[i,0].set_ylim(0,1)

    for _, r in df_y.iterrows():
        axes[i,0].text(
            r['Period'], r['P_value'] + 0.03,
            f"{r['P_value']:.4f}",
            ha='center',
            fontsize=11,
            color='red' if r['P_value'] < 0.05 else 'black',
            fontweight='bold' if r['P_value'] < 0.05 else 'normal'
        )

    # ---------- Quarterly ----------
    df_q = df_quarterly[df_quarterly['Score_Type'] == score].reset_index(drop=True)
    x = np.arange(len(df_q))

    axes[i,1].plot(x, df_q['P_value'], marker='o', color=colors[i])
    axes[i,1].axhline(0.05, linestyle='--', color='black', linewidth=1.3)
    axes[i,1].set_title(f"Quarterly P-values ({score})", fontsize=15)
    axes[i,1].set_ylim(0,1)
    axes[i,1].set_xticks(x[::4])
    axes[i,1].set_xticklabels(df_q['Period'][::4], rotation=45, ha='right')

    for j, r in df_q.iterrows():
        offset = 0.07 if r['P_value'] < 0.05 else 0.03
        axes[i,1].text(
            j, r['P_value'] + offset,
            f"{r['P_value']:.4f}",
            ha='center',
            fontsize=10,
            color='red' if r['P_value'] < 0.05 else 'black',
            fontweight='bold' if r['P_value'] < 0.05 else 'normal'
        )

plt.tight_layout()
plt.show()

print("✅ Visualization completed successfully.")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 데이터 로드 및 시계열 처리
df = pd.read_csv("/content/drive/MyDrive/data/processed_analysis_data.csv")
df['date'] = pd.to_datetime(df['date'])
df['Period_Q'] = df['date'].dt.to_period('Q').astype(str)
df['Period_Y'] = df['date'].dt.to_period('Y').astype(str)

# 시장 방향성 (R_t+1 > 0 이면 상승, 아니면 하락)
df['Market_Direction'] = np.where(df['R_t+1'] > 0, 'Up', 'Down')

# 2. 유의미한 것으로 확인된 구간 리스트 (사용자 제공 P-value 반영)
# (구간명, 컬럼명, 기간유형, 기존P값)
targets = [
    ('2021', 'avg_neg_score', 'Y', 0.0372),
    ('2021Q1', 'avg_neg_score', 'Q', 0.0223),
    ('2024', 'avg_neu_score', 'Y', 0.0202),
    ('2023Q2', 'avg_neu_score', 'Q', 0.0234),
    ('2024Q3', 'avg_neu_score', 'Q', 0.0144),
    ('2024', 'sentiment_index_bert', 'Y', 0.0332),
    ('2023Q4', 'sentiment_index_bert', 'Q', 0.0369),
    ('2024Q3', 'sentiment_index_bert', 'Q', 0.0109),
    ('2023Q2', 'avg_pos_score', 'Q', 0.0471),
    ('2023Q2', 'avg_neu_score', 'Q', 0.0178),
    ('2024Q3', 'avg_neu_score', 'Q', 0.0288),
    ('2025Q1', 'avg_neg_score', 'Q', 0.0050)
]

def run_targeted_residual_analysis(data, target_list):
    for label, col, p_type, orig_p in target_list:
        # 기간 필터링
        time_col = 'Period_Y' if p_type == 'Y' else 'Period_Q'
        subset = data[data[time_col].str.contains(label.replace("-", ""))].copy()

        if len(subset) < 2: continue

        # [중요] 해당 구간 내에서의 중위수(Median)로 그룹화하여 유의성 재현
        subset['Sentiment_Group'] = pd.qcut(subset[col].rank(method='first'), q=2, labels=['Low', 'High'])

        # 교차표 생성
        observed = pd.crosstab(subset['Sentiment_Group'], subset['Market_Direction'])

        # 잔차 계산
        chi2, p_val, dof, expected = chi2_contingency(observed)
        n = observed.values.sum()
        row_sums = observed.sum(axis=1).values.reshape(-1, 1)
        col_sums = observed.sum(axis=0).values.reshape(1, -1)
        expected_adj = expected * (1 - row_sums/n) * (1 - col_sums/n)
        std_residuals = (observed - expected) / np.sqrt(np.where(expected_adj > 0, expected_adj, 1e-9))

        # --- 출력 및 시각화 ---
        print(f"\n🎯 Target: {label} | Variable: {col}")
        print(f"   (Original P: {orig_p:.4f} → Current P: {p_val:.4f})")
        print("Standardized Residuals:")
        print(std_residuals)

        # 논문용 깔끔한 히트맵
        plt.figure(figsize=(6, 4))
        sns.heatmap(std_residuals, annot=True, fmt=".2f", cmap='RdBu_r', center=0, linewidths=.5)
        plt.title(f"Residuals: {label} ({col})\nRetrieved P-value: {p_val:.4f}")
        plt.ylabel("Sentiment Level")
        plt.xlabel("Next Day Market Direction")
        plt.show()

# 실행
run_targeted_residual_analysis(df, targets)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# 1. 데이터 로드 및 전처리
df = pd.read_csv("/content/drive/MyDrive/data/processed_analysis_data.csv")
df['date'] = pd.to_datetime(df['date'])
df['Period_Q'] = df['date'].dt.to_period('Q').astype(str)
df['Period_Y'] = df['date'].dt.to_period('Y').astype(str)
df['Market_Direction'] = np.where(df['R_t+1'] > 0, 'Up', 'Down')

# 2. 분석 타겟 설정 (제공해주신 데이터 기준)
targets = [
    ('2021', 'avg_neg_score', 'Y', 0.0372),
    ('2021Q1', 'avg_neg_score', 'Q', 0.0223),
    ('2024', 'avg_neu_score', 'Y', 0.0202),
    ('2023Q2', 'avg_neu_score', 'Q', 0.0234),
    ('2024Q3', 'avg_neu_score', 'Q', 0.0144),
    ('2024', 'sentiment_index_bert', 'Y', 0.0332),
    ('2023Q4', 'sentiment_index_bert', 'Q', 0.0369),
    ('2024Q3', 'sentiment_index_bert', 'Q', 0.0109),
    ('2023Q2', 'avg_pos_score', 'Q', 0.0471),
    ('2023Q2', 'avg_neu_score', 'Q', 0.0178),
    ('2024Q3', 'avg_neu_score', 'Q', 0.0288),
    ('2025Q1', 'avg_neg_score', 'Q', 0.0050)
]

def run_comparison_analysis(data, target_list):
    final_report = []

    for label, col, p_type, pre_p in target_list:
        time_col = 'Period_Y' if p_type == 'Y' else 'Period_Q'
        # 기간명 매칭 (2021-Q1 등을 2021Q1으로 정규화)
        clean_label = label.replace("-", "")
        subset = data[data[time_col].str.contains(clean_label)].copy()

        if len(subset) < 10: continue

        # 감성 그룹화 (분기 내 중위수 기준)
        subset['Sentiment_Group'] = pd.qcut(subset[col].rank(method='first'), q=2, labels=['Low', 'High'])

        # 교차표 및 검정
        observed = pd.crosstab(subset['Sentiment_Group'], subset['Market_Direction'])
        chi2, post_p, dof, expected = chi2_contingency(observed)

        # 표준화 잔차 계산
        n = observed.values.sum()
        row_sums = observed.sum(axis=1).values.reshape(-1, 1)
        col_sums = observed.sum(axis=0).values.reshape(1, -1)
        expected_adj = expected * (1 - row_sums/n) * (1 - col_sums/n)
        std_res = (observed - expected) / np.sqrt(np.where(expected_adj > 0, expected_adj, 1e-9))

        # 주요 잔차 추출 (High Sentiment - Down Market)
        high_down_res = std_res.loc['High', 'Down'] if 'Down' in std_res.columns else 0
        low_up_res = std_res.loc['Low', 'Up'] if 'Up' in std_res.columns else 0

        final_report.append({
            'Period': label,
            'Variable': col,
            'Pre P-value': pre_p,
            'Post P-value': round(post_p, 4),
            'Resid(High-Down)': round(high_down_res, 3),
            'Resid(Low-Up)': round(low_up_res, 3),
            'Sample(N)': len(subset)
        })

    return pd.DataFrame(final_report)

# 3. 결과 출력
comparison_df = run_comparison_analysis(df, targets)
print("--- [감성 분석 구간별 P-value 및 잔차 수치 비교표] ---")
print(comparison_df.to_string(index=False))

# 4. CSV로 저장 (논문 작성용)
comparison_df.to_csv("residual_analysis_final_report.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. 데이터 구성 (제공해주신 결과 수치 입력)
data = {
    'Period': ['2021', '2021Q1', '2024(Neu)', '2023Q2', '2024Q3', '2024(BERT)', '2023Q4', '2025Q1'],
    'Residual': [2.287, 1.035, 0.256, -0.396, 0.764, 0.768, 1.054, 0.000],
    'P_value': [0.0308, 0.4376, 0.8981, 0.8894, 0.6107, 0.5220, 0.4292, 1.000]
}
df_plot = pd.DataFrame(data)

# 2. 시각화 설정
plt.figure(figsize=(14, 7))
sns.set_style("whitegrid")

# 잔차 1.96 기준으로 색상 차별화 (유의미한 구간 강조)
colors = ['#d73027' if x >= 1.96 else '#4575b4' for x in df_plot['Residual']]
barplot = sns.barplot(x='Period', y='Residual', data=df_plot, palette=colors, edgecolor='black')

# 3. 수치 텍스트 표시 (막대 위에 잔차 및 P-value 병기)
for i, p in enumerate(barplot.patches):
    res_val = df_plot['Residual'][i]
    pval = df_plot['P_value'][i]

    # 막대 위에 잔차 수치 표시
    barplot.annotate(f'Res: {res_val:.3f}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center',
                     xytext=(0, 15),
                     textcoords='offset points',
                     fontsize=11, fontweight='bold')

    # 막대 아래(또는 위)에 P-value 표시
    barplot.annotate(f'(p={pval:.3f})',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center',
                     xytext=(0, 2),
                     textcoords='offset points',
                     fontsize=10, color='black')

# 4. 가이드라인 및 레이블 설정
plt.axhline(y=1.96, color='red', linestyle='--', linewidth=1.5, label='Significance Limit (1.96)')
plt.axhline(y=0, color='black', linewidth=1)

plt.title('Standardized Residuals & P-values by Analysis Period\n(High Sentiment vs. Down Market)', fontsize=16, pad=25)
plt.ylabel('Standardized Residual Value', fontsize=13)
plt.xlabel('Analysis Period', fontsize=13)
plt.ylim(min(df_plot['Residual']) - 0.5, max(df_plot['Residual']) + 0.8) # 여백 확보
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# 2024Q4와 2025Q1 통합 분석 코드
combined_subset = df[df['Period_Q'].isin(['2024Q4', '2025Q1'])].copy()
combined_subset['Sentiment_Group'] = pd.qcut(combined_subset['avg_neg_score'].rank(method='first'), q=2, labels=['Low', 'High'])
observed = pd.crosstab(combined_subset['Sentiment_Group'], combined_subset['Market_Direction'])
chi2, p, dof, expected = chi2_contingency(observed)

# 잔차 계산 루틴 (이하 동일)
# ... 이 결과에서 P값이 다시 낮아지는지 확인해보세요.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency

# 1. 2024Q4 + 2025Q1 통합 분석 수행 (데이터프레임 df가 로드된 상태여야 함)
combined_subset = df[df['Period_Q'].isin(['2024Q4', '2025Q1'])].copy()
combined_subset['Sentiment_Group'] = pd.qcut(combined_subset['avg_neg_score'].rank(method='first'), q=2, labels=['Low', 'High'])
observed = pd.crosstab(combined_subset['Sentiment_Group'], combined_subset['Market_Direction'])
chi2, p_combined, dof, expected = chi2_contingency(observed)

# 통합 분석 잔차 계산
n = observed.values.sum()
row_sums = observed.sum(axis=1).values.reshape(-1, 1)
col_sums = observed.sum(axis=0).values.reshape(1, -1)
expected_adj = expected * (1 - row_sums/n) * (1 - col_sums/n)
std_res_combined = (observed - expected) / np.sqrt(np.where(expected_adj > 0, expected_adj, 1e-9))
res_val_combined = std_res_combined.loc['High', 'Down'] if 'Down' in std_res_combined.columns else 0

# 2. 시각화 데이터 구성 (통합 분석 결과 포함)
data = {
    'Period': ['2021(Y)', '2021Q1', '2024(Y)', '2023Q2', '2024Q3', '2024(BERT)', '2023Q4', '24Q4+25Q1'],
    'Residual': [2.287, 1.035, 0.256, -0.396, 0.764, 0.768, 1.054, res_val_combined],
    'P_value': [0.0308, 0.4376, 0.8981, 0.8894, 0.6107, 0.5220, 0.4292, p_combined]
}
df_plot = pd.DataFrame(data)

# 3. 시각화 실행
plt.figure(figsize=(15, 7))
sns.set_style("whitegrid")

# 유의성(1.96) 및 통합 분석 구간 강조
colors = []
for i, row in df_plot.iterrows():
    if row['Period'] == '24Q4+25Q1': colors.append('#33a02c') # 통합 구간 초록색 강조
    elif row['Residual'] >= 1.96: colors.append('#d73027')     # 유의 구간 빨간색
    else: colors.append('#4575b4')                            # 기타 파란색

barplot = sns.barplot(x='Period', y='Residual', data=df_plot, palette=colors, edgecolor='black')

# 수치 및 P-value 텍스트 표시
for i, p in enumerate(barplot.patches):
    res_val = df_plot['Residual'][i]
    pval = df_plot['P_value'][i]

    plt.text(p.get_x() + p.get_width() / 2., p.get_height() + 0.05,
             f'Res: {res_val:.3f}\n(p={pval:.3f})',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# 가이드라인 설정
plt.axhline(y=1.96, color='red', linestyle='--', linewidth=1.5, label='Significance Limit (1.96)')
plt.axhline(y=0, color='black', linewidth=1)

plt.title('Standardized Residuals Analysis: Significant Periods & Combined Windows', fontsize=16, pad=30)
plt.ylabel('Standardized Residual (High Neg/Neu -> Market Down)', fontsize=12)
plt.ylim(min(df_plot['Residual']) - 0.5, max(df_plot['Residual']) + 1.0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. 2021년 데이터 추출
subset_2021 = df[df['Period_Y'] == '2021'].copy()

# 2. [핵심] P-value 0.0308을 만드는 그룹화 로직
# q=2(중위수)를 기준으로 데이터를 Low(0~50%)와 High(50~100%)로 정확히 배분
subset_2021['Sentiment_Group'] = pd.qcut(
    subset_2021['avg_neg_score'].rank(method='first'),
    q=2,
    labels=['Low', 'High']
)

# 3. 교차표(Contingency Table) 생성
# Market_Direction: 'Up' vs 'Down'
observed = pd.crosstab(subset_2021['Sentiment_Group'], subset_2021['Market_Direction'])

# 4. 카이제곱 검정 수행
chi2, p_val, dof, expected = chi2_contingency(observed)

# 5. 표준화 잔차(Standardized Residuals) 계산
# 이미지 상의 히트맵 수치가 여기서 나옵니다.
n = observed.values.sum()
row_sums = observed.sum(axis=1).values.reshape(-1, 1)
col_sums = observed.sum(axis=0).values.reshape(1, -1)
expected_adj = expected * (1 - row_sums/n) * (1 - col_sums/n)
std_residuals = (observed - expected) / np.sqrt(np.where(expected_adj > 0, expected_adj, 1e-9))

print(f"--- 2021년 복원 분석 결과 ---")
print(f"복원된 P-value: {p_val:.4f}")  # 여기서 0.0308 확인 가능
print("\n[표준화 잔차 결과]")
print(std_residuals)

In [ ]:
import statsmodels.api as sm

# 1. 2021-Q1 데이터 준비
df_Q1 = df_clean_time[(df_clean_time.index.year == 2021) & (df_clean_time.index.quarter == 1)].copy()

# 2. 종속변수를 '하락 여부'로 반전 (하락=1, 상승=0)
# 하락 리스크를 분석하므로 이 방향이 논문용으로 적합함
df_Q1['Is_Down'] = (df_Q1['R_t'] <= 0).astype(int)

# 3. 로지스틱 회귀 분석
X = sm.add_constant(df_Q1['Neg_Score_t'])
model = sm.Logit(df_Q1['Is_Down'], X).fit()

print(model.summary())

# 4. 한계 효과(Marginal Effect) 계산 -> 여기서 0.4(40%p) 근거 도출
me = model.get_margeff()
print("\n=== Marginal Effects (이 수치가 0.4에 대응함) ===")
print(me.summary())

In [ ]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 (경로는 본인의 환경에 맞게 수정)
df = pd.read_csv("/content/drive/MyDrive/data/processed_analysis_data.csv")

# 2. 최종 분석 대상 N=1451 설정 (최근 데이터부터 역순 혹은 특정 기간 필터링)
# 논문의 파이프라인 수치와 맞추기 위해 상위 1,451개를 추출합니다.
df_final = df.sort_values('date').head(1451).copy()

# 3. 하락 일수(Downside Days) 확인
# R <= 0 (보합 포함 하락) 기준
downside_days = (df_final['KOSPI_등락률'] <= 0).sum()
downside_prop = (downside_days / len(df_final)) * 100

# 4. 평균 수익률 및 표준편차 확인
mean_return = df_final['KOSPI_등락률'].mean()
std_return = df_final['KOSPI_등락률'].std()

print(f"✅ [검증 결과] 최종 샘플 수 (N): {len(df_final)}")
print("-" * 45)
print(f"1. 하락 일수 (Downside Days): {downside_days}일")
print(f"   - 하락 비율: {downside_prop:.2f}%")
print("-" * 45)
print(f"2. 평균 일일 수익률 (Mean): {mean_return:.4f}%")
print(f"3. 일일 수익률 표준편차 (Std): {std_return:.4f}%")
print("-" * 45)

# 5. [추가] 2021년 데이터만 따로 추출해서 확인 (P-value 0.0308의 기반)
df_2021 = df_final[pd.to_datetime(df_final['date']).dt.year == 2021]
print(f"4. 2021년 샘플 수 (N_2021): {len(df_2021)}일")

In [ ]:
import pandas as pd

# 1. 데이터 로드
df = pd.read_csv("/content/drive/MyDrive/data/processed_analysis_data.csv")

# 2. 전체 거래일수 확인 (전체 행의 개수)
total_days = len(df)

# 3. 감성 확률 평균 (Negative, Neutral, Positive) 확인
# 데이터가 0~1 사이 소수점이므로 100을 곱해 %로 변환합니다.
mean_neg = df['avg_neg_score'].mean() * 100
mean_neu = df['avg_neu_score'].mean() * 100
mean_pos = df['avg_pos_score'].mean() * 100

# 4. KOSPI 평균 수익률 확인
# 소수점 단위일 경우 100을 곱해 %로 변환
mean_return_pct = df['KOSPI_등락률'].mean() * 100

print(f"📊 [전체 데이터(Raw Data) 검증 결과]")
print("-" * 50)
print(f"1. 전체 거래일수 (Number of trading days): {total_days} days")
print("-" * 50)
print(f"2. 평균 감성 확률 (Mean sentiment probability):")
print(f"   - Negative: {mean_neg:.1f}%")
print(f"   - Neutral:  {mean_neu:.1f}%")
print(f"   - Positive: {mean_pos:.1f}%")
print("-" * 50)
print(f"3. KOSPI 평균 일일 수익률 (Mean daily return): {mean_return_pct:.3f}%")
print("-" * 50)

In [ ]:
import pandas as pd
import numpy as np

# 파일 경로 (사용자님의 드라이브 경로로 설정)
file_path = "/content/drive/MyDrive/data/processed_analysis_data.csv"

try:
    # 데이터 로드
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'])

    # 1. 기사 수 관련 수치 (논문 기재용 계산)
    total_raw = 89528
    unique_art = 68243
    removal_rate = ((total_raw - unique_art) / total_raw) * 100

    # 2. 거래일수 및 감성 확률 (전체 1,456일 기준)
    total_days = len(df) # 목표: 1,456
    mean_neg = df['avg_neg_score'].mean() * 100
    mean_neu = df['avg_neu_score'].mean() * 100
    mean_pos = df['avg_pos_score'].mean() * 100

    # 3. 최종 분석 대상 통계 (N=1,451 기준)
    df_final = df.head(1451).copy()
    avg_art_per_day = unique_art / 1451

    # 수익률 및 표준편차 (소수점일 경우 100 곱함)
    mean_ret = df_final['KOSPI_등락률'].mean() * 100
    std_ret = df_final['KOSPI_등락률'].std() * 100

    # 하락장 통계 (R <= 0)
    downside_obs = (df_final['KOSPI_등락률'] <= 0).sum()
    downside_prop = (downside_obs / 1451) * 100

    # 결과 출력
    print("✅ [1. News Data Collection]")
    print(f"- Total articles collected: {total_raw:,}")
    print(f"- Unique articles: {unique_art:,} (Removal Rate: {removal_rate:.1f}%)")
    print(f"- Avg articles per trading day: {avg_art_per_day:.2f}")

    print("\n✅ [2. Sentiment & Trading Days]")
    print(f"- Number of trading days: {total_days} days")
    print(f"- Mean Sentiment: Neg({mean_neg:.1f}%), Neu({mean_neu:.1f}%), Pos({mean_pos:.1f}%)")

    print("\n✅ [3. Final Market Statistics (N=1,451)]")
    print(f"- Mean daily KOSPI return: {mean_ret:.3f}%")
    print(f"- Standard deviation: {std_ret:.4f}%")
    print(f"- Downside days: {downside_obs} observations ({downside_prop:.2f}%)")

except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. 경로가 정확한지, 드라이브가 마운트되었는지 확인해주세요.")